# Initializing Class Instances — Advanced Tutorial Problems with Solutions

This notebook revisits **initializing class instances**, but in a much more guided style.

Instead of jumping straight from a problem statement to a large solution, we will repeatedly use this structure:

1. understand the goal,
2. isolate the object state,
3. predict Python's behavior,
4. implement one small piece,
5. test it,
6. improve the design,
7. inspect the complete solution.

The main topics are:

- `__init__`
- instance state
- validation
- normalization
- mutable defaults
- defensive copying
- keyword-only constructor arguments
- properties and invariants
- alternate constructors
- inheritance and `super()`
- cooperative multiple inheritance
- `__new__`
- immutable subclasses
- dataclasses
- `__slots__`
- dependency injection
- constructor testing
- constructor anti-patterns


## Reminder: creation and initialization are different

When we write:

```python
obj = SomeClass(...)
```

Python may involve two separate hooks:

```text
__new__  -> creates / returns an object
__init__ -> initializes that already-created object
```

For normal user-defined classes, we usually customize `__init__`.

We only customize `__new__` when the **creation phase itself** must change.


In [1]:
class Person:
    def __init__(self, name):
        self.name = name

p = Person("Eric")
print(p.__dict__)


{'name': 'Eric'}


By the time `Person.__init__` runs, `self` is already a real `Person` instance.

That idea will be important in almost every problem below.


# Problem 1 — Trace object creation and initialization

We want to make the lifecycle visible.

We will build a class that prints messages from both:

```python
__new__
__init__
```

Before running any code, predict the order.

Which one must happen first?


## Step 1 — Reason about responsibilities

`__init__` receives `self`.

That means an object must already exist before `__init__` can receive it.

So we expect:

```text
1. __new__
2. __init__
```


## Step 2 — Implement only `__new__`

For a normal class, the standard way to ask the superclass to create the raw instance is:

```python
super().__new__(cls)
```


In [2]:
class LifecycleDemo:
    def __new__(cls, value):
        print("__new__ started")
        instance = super().__new__(cls)
        print("__new__ created:", instance)
        return instance


In [3]:
obj = LifecycleDemo(10)
print(type(obj))
print(obj.__dict__)


__new__ started
__new__ created: <__main__.LifecycleDemo object at 0x000002C8B2DDE7B0>
<class '__main__.LifecycleDemo'>
{}


There is no custom `__init__` yet.

The object is still created because `__new__` returned an instance.


## Step 3 — Add `__init__`

Now we initialize the object's state.


In [4]:
class LifecycleDemo:
    def __new__(cls, value):
        print("1. __new__")
        instance = super().__new__(cls)
        print("2. id in __new__:", id(instance))
        return instance

    def __init__(self, value):
        print("3. __init__")
        print("4. id in __init__:", id(self))
        self.value = value


In [5]:
obj = LifecycleDemo(10)

print("5. final id:", id(obj))
print("6. final state:", obj.__dict__)


1. __new__
2. id in __new__: 3061017602304
3. __init__
4. id in __init__: 3061017602304
5. final id: 3061017602304
6. final state: {'value': 10}


## What did we prove?

The identity is the same in all three places:

- inside `__new__`,
- inside `__init__`,
- in the final variable.

So `__init__` does not create a second object.

It initializes the object returned by `__new__`.


# Problem 2 — Protect invariants during initialization

We want a `BankTransfer`.

A valid transfer must have:

- non-empty sender,
- non-empty recipient,
- positive amount,
- non-empty currency code.

Our goal is:

> If construction succeeds, the object is already valid.


## Step 1 — Start with a naive constructor


In [6]:
class BadBankTransfer:
    def __init__(self, sender, recipient, amount, currency):
        self.sender = sender
        self.recipient = recipient
        self.amount = amount
        self.currency = currency


This constructor allows invalid state.


In [7]:
bad = BadBankTransfer("", "", -500, "")
print(bad.__dict__)


{'sender': '', 'recipient': '', 'amount': -500, 'currency': ''}


The class is syntactically correct, but the object makes no sense.

That means our constructor is not enforcing the model's invariant.


## Step 2 — Validate text values

A useful pattern for required text is:

```python
if not isinstance(value, str):
    raise TypeError(...)

if not value.strip():
    raise ValueError(...)
```

Why call `.strip()`?

Because a string containing only spaces should usually count as empty.


## Step 3 — Validate numeric values

The transfer amount should be numeric and greater than zero.

We will also deliberately reject booleans.

Remember:

```python
isinstance(True, int)
```

is `True` in Python.


In [8]:
print(isinstance(True, int))
print(isinstance(False, int))


True
True


## Step 4 — Complete solution


In [9]:
class BankTransfer:
    def __init__(self, sender, recipient, amount, currency):
        if not isinstance(sender, str):
            raise TypeError("sender must be a string")
        if not sender.strip():
            raise ValueError("sender cannot be empty")

        if not isinstance(recipient, str):
            raise TypeError("recipient must be a string")
        if not recipient.strip():
            raise ValueError("recipient cannot be empty")

        if not isinstance(amount, (int, float)) or isinstance(amount, bool):
            raise TypeError("amount must be numeric")
        if amount <= 0:
            raise ValueError("amount must be positive")

        if not isinstance(currency, str):
            raise TypeError("currency must be a string")

        currency = currency.strip().upper()

        if len(currency) != 3 or not currency.isalpha():
            raise ValueError("currency must contain exactly 3 letters")

        self.sender = sender.strip()
        self.recipient = recipient.strip()
        self.amount = float(amount)
        self.currency = currency


In [10]:
transfer = BankTransfer(
    "  Alice  ",
    "  Bob  ",
    125.50,
    " usd ",
)

print(transfer.__dict__)


{'sender': 'Alice', 'recipient': 'Bob', 'amount': 125.5, 'currency': 'USD'}


## Step 5 — Test failure cases

Testing only successful construction is not enough.

A constructor's error behavior is part of its API.


In [11]:
cases = [
    ("", "Bob", 10, "USD"),
    ("Alice", "", 10, "USD"),
    ("Alice", "Bob", 0, "USD"),
    ("Alice", "Bob", 10, "US"),
]

for args in cases:
    try:
        BankTransfer(*args)
    except (TypeError, ValueError) as exc:
        print(args, "->", type(exc).__name__, exc)


('', 'Bob', 10, 'USD') -> ValueError sender cannot be empty
('Alice', '', 10, 'USD') -> ValueError recipient cannot be empty
('Alice', 'Bob', 0, 'USD') -> ValueError amount must be positive
('Alice', 'Bob', 10, 'US') -> ValueError currency must contain exactly 3 letters


## Design takeaway

A robust constructor often follows this order:

```text
validate
normalize
store
```

That helps ensure that a successfully created instance starts in a meaningful state.


# Problem 3 — Mutable default arguments

We want a `TodoList` with an optional list of tasks.

A tempting constructor is:


In [12]:
class BadTodoList:
    def __init__(self, tasks=[]):
        self.tasks = tasks


## Step 1 — Predict the bug

Create two instances:

```python
a = BadTodoList()
b = BadTodoList()
```

Then modify only `a`.

Should `b` change?

Logically, no.

Let's test.


In [13]:
a = BadTodoList()
b = BadTodoList()

a.tasks.append("Learn __init__")

print("a:", a.tasks)
print("b:", b.tasks)
print("same list?", a.tasks is b.tasks)


a: ['Learn __init__']
b: ['Learn __init__']
same list? True


Both instances share the same list.

The default list was created once when the function definition executed.

It was not recreated for every constructor call.


## Step 2 — Use `None` as a sentinel

A standard constructor pattern is:

```python
def __init__(self, tasks=None):
    if tasks is None:
        tasks = []
```


In [14]:
class TodoList:
    def __init__(self, tasks=None):
        if tasks is None:
            tasks = []

        self.tasks = tasks


In [15]:
a = TodoList()
b = TodoList()

a.tasks.append("Learn __init__")

print("a:", a.tasks)
print("b:", b.tasks)
print("same list?", a.tasks is b.tasks)


a: ['Learn __init__']
b: []
same list? False


We fixed one problem.

But there is another question:

> If the caller passes a list, should the object share that exact list?


## Step 3 — Observe aliasing with caller-owned data


In [16]:
source = ["Task A", "Task B"]
todo = TodoList(source)

source.append("Task C")

print("source:", source)
print("todo:", todo.tasks)
print("same object?", source is todo.tasks)


source: ['Task A', 'Task B', 'Task C']
todo: ['Task A', 'Task B', 'Task C']
same object? True


If the class is supposed to own its own collection, this is undesirable.

We should copy the iterable.


## Step 4 — Final solution


In [17]:
class TodoList:
    def __init__(self, tasks=None):
        self.tasks = [] if tasks is None else list(tasks)


In [18]:
source = ["Task A", "Task B"]
todo = TodoList(source)

source.append("Task C")
todo.tasks.append("Task D")

print("source:", source)
print("todo:", todo.tasks)
print("same object?", source is todo.tasks)


source: ['Task A', 'Task B', 'Task C']
todo: ['Task A', 'Task B', 'Task D']
same object? False


## Design takeaway

There are two separate concerns:

1. avoid reusing a mutable default object,
2. decide whether incoming mutable data should be shared or copied.

Good constructor design makes ownership explicit.


# Problem 4 — Keyword-only constructor arguments

Suppose we are configuring an API client.

We need:

- base URL,
- timeout,
- retries,
- TLS verification,
- compression.

A call like this is hard to read:

```python
ApiClient("https://api.example.com", 10, 3, True, False)
```

What does each value mean?


## Step 1 — Make optional configuration keyword-only

A `*` in the signature means all later arguments must be passed by keyword.


In [19]:
class ApiClient:
    def __init__(
        self,
        base_url,
        *,
        timeout=10,
        retries=3,
        verify_tls=True,
        compression=True,
    ):
        self.base_url = base_url
        self.timeout = timeout
        self.retries = retries
        self.verify_tls = verify_tls
        self.compression = compression


In [20]:
client = ApiClient(
    "https://api.example.com",
    timeout=5,
    retries=2,
    verify_tls=True,
    compression=False,
)

print(client.__dict__)


{'base_url': 'https://api.example.com', 'timeout': 5, 'retries': 2, 'verify_tls': True, 'compression': False}


The call now documents itself.


## Step 2 — Confirm positional misuse is rejected


In [21]:
try:
    ApiClient("https://api.example.com", 5, 2)
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: ApiClient.__init__() takes 2 positional arguments but 4 were given


## Step 3 — Add constructor validation


In [22]:
class ApiClient:
    def __init__(
        self,
        base_url,
        *,
        timeout=10,
        retries=3,
        verify_tls=True,
        compression=True,
    ):
        if not isinstance(base_url, str):
            raise TypeError("base_url must be a string")

        if not base_url.startswith(("http://", "https://")):
            raise ValueError("base_url must be an HTTP(S) URL")

        if timeout <= 0:
            raise ValueError("timeout must be positive")

        if not isinstance(retries, int) or isinstance(retries, bool):
            raise TypeError("retries must be an integer")

        if retries < 0:
            raise ValueError("retries cannot be negative")

        self.base_url = base_url.rstrip("/")
        self.timeout = float(timeout)
        self.retries = retries
        self.verify_tls = bool(verify_tls)
        self.compression = bool(compression)


## Design takeaway

Keyword-only parameters are especially useful when:

- several values have the same type,
- boolean flags would otherwise appear as mysterious `True` / `False`,
- many options have defaults,
- argument order should not be part of the public API.


# Problem 5 — Derived state and synchronization

We want a `Circle`.

It needs a radius.

We also want to expose:

- diameter,
- circumference.

Should we store all three values during initialization?


## Step 1 — Try storing derived values


In [23]:
from math import pi


class StoredCircle:
    def __init__(self, radius):
        self.radius = radius
        self.diameter = 2 * radius
        self.circumference = 2 * pi * radius


In [24]:
c = StoredCircle(2)
print(c.__dict__)

c.radius = 10
print(c.__dict__)


{'radius': 2, 'diameter': 4, 'circumference': 12.566370614359172}
{'radius': 10, 'diameter': 4, 'circumference': 12.566370614359172}


The object is now inconsistent.

The radius changed, but the derived values did not.


## Step 2 — Keep only one source of truth

Store the radius.

Compute the other values when requested.


In [25]:
from math import pi


class Circle:
    def __init__(self, radius):
        if radius <= 0:
            raise ValueError("radius must be positive")

        self.radius = float(radius)

    @property
    def diameter(self):
        return 2 * self.radius

    @property
    def circumference(self):
        return 2 * pi * self.radius


In [26]:
c = Circle(2)

print(c.diameter)
print(c.circumference)

c.radius = 10

print(c.diameter)
print(c.circumference)


4.0
12.566370614359172
20
62.83185307179586


## Design takeaway

If a value is:

- fully derived from other state,
- cheap to compute,
- required to remain synchronized,

a property may be safer than storing duplicated state.


# Problem 6 — Reuse validation through a property setter

We want a `VolumeLevel`.

The value must always be from `0` to `100`.

That rule applies:

- during construction,
- during later assignment.

We should avoid writing the same validation twice.


## Step 1 — Put the invariant in the property setter


In [27]:
class VolumeLevel:
    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, new_value):
        if not isinstance(new_value, (int, float)) or isinstance(new_value, bool):
            raise TypeError("volume must be numeric")

        if not 0 <= new_value <= 100:
            raise ValueError("volume must be between 0 and 100")

        self._value = float(new_value)


## Step 2 — Initialize through the property

Inside `__init__`, assigning to `self.value` deliberately calls the setter.


In [28]:
class VolumeLevel:
    def __init__(self, value):
        self.value = value

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, new_value):
        if not isinstance(new_value, (int, float)) or isinstance(new_value, bool):
            raise TypeError("volume must be numeric")

        if not 0 <= new_value <= 100:
            raise ValueError("volume must be between 0 and 100")

        self._value = float(new_value)


In [29]:
volume = VolumeLevel(75)
print(volume.value)

volume.value = 25
print(volume.value)


75.0
25.0


## Step 3 — Test initialization and mutation with invalid values


In [30]:
for invalid in (-1, 101, "loud"):
    try:
        VolumeLevel(invalid)
    except (TypeError, ValueError) as exc:
        print("constructor:", repr(invalid), "->", type(exc).__name__, exc)

volume = VolumeLevel(50)

try:
    volume.value = 500
except ValueError as exc:
    print("assignment:", type(exc).__name__, exc)


constructor: -1 -> ValueError volume must be between 0 and 100
constructor: 101 -> ValueError volume must be between 0 and 100
constructor: 'loud' -> TypeError volume must be numeric
assignment: ValueError volume must be between 0 and 100


## Design takeaway

Centralizing an invariant prevents validation rules from drifting apart.

The constructor becomes simpler because it reuses the same public state rule.


# Problem 7 — Alternate constructors with `@classmethod`

We want a `Duration`.

Its main representation will be total seconds:

```python
Duration(5400)
```

But we also want:

```python
Duration.from_hours_minutes(1, 30)
```

We should avoid duplicating final validation.


## Step 1 — Create the primary constructor


In [31]:
class Duration:
    def __init__(self, total_seconds):
        if not isinstance(total_seconds, (int, float)) or isinstance(total_seconds, bool):
            raise TypeError("total_seconds must be numeric")

        if total_seconds < 0:
            raise ValueError("total_seconds cannot be negative")

        self.total_seconds = float(total_seconds)


## Step 2 — Convert alternate input into primary input

Hours and minutes become:

```python
hours * 3600 + minutes * 60
```

Then delegate to:

```python
cls(total_seconds)
```


In [32]:
class Duration:
    def __init__(self, total_seconds):
        if not isinstance(total_seconds, (int, float)) or isinstance(total_seconds, bool):
            raise TypeError("total_seconds must be numeric")

        if total_seconds < 0:
            raise ValueError("total_seconds cannot be negative")

        self.total_seconds = float(total_seconds)

    @classmethod
    def from_hours_minutes(cls, hours, minutes):
        total_seconds = hours * 3600 + minutes * 60
        return cls(total_seconds)


In [33]:
d = Duration.from_hours_minutes(1, 30)
print(d.total_seconds)


5400.0


## Step 3 — Why `cls(...)` instead of `Duration(...)`?

Because alternate constructors should usually cooperate with subclasses.


In [34]:
class SpecialDuration(Duration):
    pass

d = SpecialDuration.from_hours_minutes(2, 15)

print(type(d))
print(d.total_seconds)


<class '__main__.SpecialDuration'>
8100.0


## Design takeaway

A strong alternate-constructor pattern is:

```text
parse / transform
      ↓
delegate to cls(...)
      ↓
reuse the primary invariant
```


# Problem 8 — Parse text into an initialized object

Create a `Version` class.

The main constructor accepts three integers:

```python
Version(3, 12, 1)
```

We also want:

```python
Version.from_string("3.12.1")
```

The parser should not bypass normal validation.


## Step 1 — Define valid component rules

Each component must be a non-negative integer.


In [35]:
class Version:
    def __init__(self, major, minor, patch):
        for name, value in [
            ("major", major),
            ("minor", minor),
            ("patch", patch),
        ]:
            if not isinstance(value, int) or isinstance(value, bool):
                raise TypeError(f"{name} must be an integer")

            if value < 0:
                raise ValueError(f"{name} cannot be negative")

        self.major = major
        self.minor = minor
        self.patch = patch


## Step 2 — Parse the external representation

For:

```text
3.12.1
```

we split on `.`.

We expect exactly three pieces.


In [36]:
text = "3.12.1"
print(text.split("."))


['3', '12', '1']


## Step 3 — Delegate the parsed components


In [37]:
class Version:
    def __init__(self, major, minor, patch):
        for name, value in [
            ("major", major),
            ("minor", minor),
            ("patch", patch),
        ]:
            if not isinstance(value, int) or isinstance(value, bool):
                raise TypeError(f"{name} must be an integer")

            if value < 0:
                raise ValueError(f"{name} cannot be negative")

        self.major = major
        self.minor = minor
        self.patch = patch

    @classmethod
    def from_string(cls, text):
        if not isinstance(text, str):
            raise TypeError("version text must be a string")

        parts = text.split(".")

        if len(parts) != 3:
            raise ValueError("expected MAJOR.MINOR.PATCH")

        try:
            major, minor, patch = map(int, parts)
        except ValueError as exc:
            raise ValueError("version components must be integers") from exc

        return cls(major, minor, patch)

    def __repr__(self):
        return f"Version({self.major}, {self.minor}, {self.patch})"


In [38]:
v = Version.from_string("3.12.1")
print(v)


Version(3, 12, 1)


## Step 4 — Test invalid formats


In [39]:
for text in ["3.12", "3.x.1", "3.12.1.9"]:
    try:
        Version.from_string(text)
    except ValueError as exc:
        print(repr(text), "->", exc)


'3.12' -> expected MAJOR.MINOR.PATCH
'3.x.1' -> version components must be integers
'3.12.1.9' -> expected MAJOR.MINOR.PATCH


## Design takeaway

Parsing belongs in the alternate constructor.

The final object invariant still belongs in the main constructor.


# Problem 9 — Subclass initialization with `super()`

We will build:

```text
UserAccount
    ↑
AdminAccount
```

The base class owns:

- username,
- email.

The subclass adds:

- permissions.

The subclass should not duplicate base initialization logic.


## Step 1 — Build the base constructor


In [40]:
class UserAccount:
    def __init__(self, username, email):
        if not isinstance(username, str) or not username.strip():
            raise ValueError("username cannot be empty")

        if not isinstance(email, str) or email.count("@") != 1:
            raise ValueError("email must contain exactly one @")

        self.username = username.strip().lower()
        self.email = email.strip().lower()


## Step 2 — Why copying base logic is fragile

This is tempting:

```python
class AdminAccount(UserAccount):
    def __init__(...):
        self.username = ...
        self.email = ...
```

But then validation rules are duplicated.

If the base class changes, the subclass can silently become inconsistent.


## Step 3 — Delegate inherited initialization with `super()`


In [41]:
class AdminAccount(UserAccount):
    def __init__(self, username, email, permissions=None):
        super().__init__(username, email)

        self.permissions = (
            set()
            if permissions is None
            else set(permissions)
        )


In [42]:
admin = AdminAccount(
    "  RootUser  ",
    " ROOT@EXAMPLE.COM ",
    permissions=["read", "write"],
)

print(admin.__dict__)


{'username': 'rootuser', 'email': 'root@example.com', 'permissions': {'read', 'write'}}


## Step 4 — Confirm base validation still applies


In [43]:
try:
    AdminAccount("", "broken-email")
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)


ValueError: username cannot be empty


## Design takeaway

A subclass constructor usually has two jobs:

1. delegate inherited state initialization,
2. initialize only the additional state introduced by the subclass.


# Problem 10 — Follow a complete initialization chain

Now consider:

```text
Entity
  ↑
NamedEntity
  ↑
Article
```

We will print messages to see exactly how control moves through the constructors.


In [44]:
class Entity:
    def __init__(self, entity_id):
        print("Entity: start")
        self.entity_id = entity_id
        print("Entity: end")


class NamedEntity(Entity):
    def __init__(self, entity_id, name):
        print("NamedEntity: before super")
        super().__init__(entity_id)
        print("NamedEntity: after super")
        self.name = name


class Article(NamedEntity):
    def __init__(self, entity_id, name, body):
        print("Article: before super")
        super().__init__(entity_id, name)
        print("Article: after super")
        self.body = body


In [45]:
article = Article(
    1,
    "Initialization",
    "Constructors establish object state.",
)

print(article.__dict__)


Article: before super
NamedEntity: before super
Entity: start
Entity: end
NamedEntity: after super
Article: after super
{'entity_id': 1, 'name': 'Initialization', 'body': 'Constructors establish object state.'}


## Read the trace carefully

The call begins with the most specific class:

```text
Article
```

Then `super()` sends control upward:

```text
NamedEntity
Entity
```

After `Entity` returns, control unwinds back down.

So the post-`super()` lines appear in reverse order.


# Problem 11 — Cooperative multiple inheritance

This is a more advanced scenario.

We want a final class with three independent initialization concerns:

- record ID,
- owner,
- creation timestamp.

We will implement them with cooperative constructors and `super()`.


## Step 1 — Understand the MRO

`super()` does not simply mean:

> call my direct parent

It means:

> continue method lookup after the current class according to the Method Resolution Order.


In [46]:
class Root:
    pass

class OwnerMixin(Root):
    pass

class TimestampMixin(Root):
    pass

class Record(OwnerMixin, TimestampMixin):
    pass

print(Record.__mro__)


(<class '__main__.Record'>, <class '__main__.OwnerMixin'>, <class '__main__.TimestampMixin'>, <class '__main__.Root'>, <class 'object'>)


## Step 2 — Build a terminal cooperative base

The final base will reject any unused keyword arguments.

That helps catch misspelled or unsupported constructor options.


In [47]:
class CooperativeBase:
    def __init__(self, **kwargs):
        if kwargs:
            unexpected = ", ".join(sorted(kwargs))
            raise TypeError(
                f"unexpected constructor arguments: {unexpected}"
            )

        super().__init__()


## Step 3 — Let each class consume only its own arguments


In [48]:
class OwnerMixin(CooperativeBase):
    def __init__(self, *, owner, **kwargs):
        if not owner:
            raise ValueError("owner cannot be empty")

        self.owner = owner
        super().__init__(**kwargs)


class TimestampMixin(CooperativeBase):
    def __init__(self, *, created_at, **kwargs):
        self.created_at = created_at
        super().__init__(**kwargs)


## Step 4 — Build the final class


In [49]:
class Record(OwnerMixin, TimestampMixin):
    def __init__(self, *, record_id, **kwargs):
        if record_id <= 0:
            raise ValueError("record_id must be positive")

        self.record_id = record_id
        super().__init__(**kwargs)


In [50]:
record = Record(
    record_id=100,
    owner="Ada",
    created_at="2026-09-11T12:00:00",
)

print(Record.__mro__)
print(record.__dict__)


(<class '__main__.Record'>, <class '__main__.OwnerMixin'>, <class '__main__.TimestampMixin'>, <class '__main__.CooperativeBase'>, <class 'object'>)
{'record_id': 100, 'owner': 'Ada', 'created_at': '2026-09-11T12:00:00'}


## Step 5 — Follow the keyword flow

At the beginning we have:

```text
record_id
owner
created_at
```

`Record` consumes:

```text
record_id
```

`OwnerMixin` consumes:

```text
owner
```

`TimestampMixin` consumes:

```text
created_at
```

Then the base receives nothing.

That is cooperative initialization.


## Design takeaway

For cooperative multiple inheritance:

- consume only your own arguments,
- forward the rest,
- use `super()`,
- avoid hard-coding a specific next parent.


# Problem 12 — Why `__init__` must return `None`

`__init__` is not a factory function.

It configures an object that already exists.

Let's deliberately violate the rule.


In [51]:
class WrongInitializer:
    def __init__(self):
        return 123


## Predict the result

Will Python:

1. ignore `123`,
2. return `123`,
3. raise an error?


In [52]:
try:
    WrongInitializer()
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: __init__() should return None, not 'int'


The answer is: Python raises a `TypeError`.

The construction result comes from the creation phase.

`__init__` must return `None`.


# Problem 13 — Return another type from `__new__`

Now we will change the creation phase itself.

We will make a class whose `__new__` returns a plain dictionary.


In [53]:
class ReturnsDictionary:
    def __new__(cls, value):
        print("__new__ called")
        return {"value": value}

    def __init__(self, value):
        print("__init__ called")
        self.value = value


## Predict two things

For:

```python
result = ReturnsDictionary(10)
```

predict:

1. the type of `result`,
2. whether `ReturnsDictionary.__init__` runs.


In [54]:
result = ReturnsDictionary(10)

print(type(result))
print(result)


__new__ called
<class 'dict'>
{'value': 10}


`result` is a `dict`.

And `ReturnsDictionary.__init__` does not run.

Because the object returned from `__new__` is not an instance of the requested class, Python does not continue with that class's initializer.


# Problem 14 — Immutable subclass initialization

We will subclass `tuple` to create a `Coordinate3D`.

This is a real case where `__new__` matters.

A tuple's contents are immutable.

That means the tuple payload must be supplied during creation.


## Step 1 — Create the tuple value inside `__new__`


In [55]:
class Coordinate3D(tuple):
    def __new__(cls, x, y, z):
        values = (float(x), float(y), float(z))
        return super().__new__(cls, values)


In [56]:
point = Coordinate3D(1, 2, 3)

print(point)
print(type(point))


(1.0, 2.0, 3.0)
<class '__main__.Coordinate3D'>


## Step 2 — Expose meaningful names


In [57]:
class Coordinate3D(tuple):
    def __new__(cls, x, y, z):
        values = (float(x), float(y), float(z))
        return super().__new__(cls, values)

    @property
    def x(self):
        return self[0]

    @property
    def y(self):
        return self[1]

    @property
    def z(self):
        return self[2]


In [58]:
point = Coordinate3D(1, 2, 3)

print(point.x)
print(point.y)
print(point.z)


1.0
2.0
3.0


## Step 3 — Add validation before immutable creation


In [59]:
class Coordinate3D(tuple):
    def __new__(cls, x, y, z):
        values = (x, y, z)

        for value in values:
            if not isinstance(value, (int, float)) or isinstance(value, bool):
                raise TypeError("coordinates must be numeric")

        normalized = tuple(float(value) for value in values)
        return super().__new__(cls, normalized)

    @property
    def x(self):
        return self[0]

    @property
    def y(self):
        return self[1]

    @property
    def z(self):
        return self[2]


## Design takeaway

For immutable built-in subclasses, the immutable payload often has to be established in `__new__`.

`__init__` is too late to replace the tuple contents.


# Problem 15 — Instance caching with `__new__`

We want a `Tag` class where normalized equal values reuse the same instance.

Example:

```python
a = Tag(" Python ")
b = Tag("python")

a is b
# True
```

This is an advanced creation pattern.


## Step 1 — Normalize the cache key

We want these inputs to represent the same tag:

```text
" Python "
"python"
"PYTHON"
```

So the cache key will be:

```python
text.strip().lower()
```


## Step 2 — Store created instances in a class-level dictionary


In [60]:
class Tag:
    _cache = {}


## Step 3 — Decide in `__new__` whether to create or reuse


In [61]:
class Tag:
    _cache = {}

    def __new__(cls, text):
        key = text.strip().lower()

        if key not in cls._cache:
            cls._cache[key] = super().__new__(cls)

        return cls._cache[key]


## Step 4 — Handle repeated initialization

Even when `__new__` returns a cached instance, `__init__` may run again.

So we make initialization safe to repeat.


In [62]:
class Tag:
    _cache = {}

    def __new__(cls, text):
        if not isinstance(text, str):
            raise TypeError("text must be a string")

        key = text.strip().lower()

        if not key:
            raise ValueError("tag cannot be empty")

        if key not in cls._cache:
            cls._cache[key] = super().__new__(cls)

        return cls._cache[key]

    def __init__(self, text):
        if getattr(self, "_initialized", False):
            return

        self.text = text.strip().lower()
        self._initialized = True

    def __repr__(self):
        return f"Tag({self.text!r})"


In [63]:
a = Tag(" Python ")
b = Tag("python")
c = Tag("oop")

print(a)
print(b)
print(c)

print("a is b:", a is b)
print("a is c:", a is c)


Tag('python')
Tag('python')
Tag('oop')
a is b: True
a is c: False


## Design caution

Caching in `__new__` raises more questions:

- cache lifetime,
- memory usage,
- thread safety,
- subclass behavior,
- repeated constructor calls.

A factory function or explicit cache is often simpler.

Use this pattern only when instance identity itself matters.


# Problem 16 — `__slots__` and initialization

A normal Python instance often stores attributes in `__dict__`.

Let's see that first.


In [64]:
class NormalVector:
    def __init__(self, x, y):
        self.x = x
        self.y = y


v = NormalVector(1, 2)
print(v.__dict__)


{'x': 1, 'y': 2}


We can also add arbitrary attributes later.


In [65]:
v.label = "demo"
print(v.__dict__)


{'x': 1, 'y': 2, 'label': 'demo'}


## Step 1 — Restrict instance attributes with `__slots__`


In [66]:
class SlottedVector:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)


In [67]:
v = SlottedVector(1, 2)

print(v.x, v.y)
print("has __dict__?", hasattr(v, "__dict__"))


1.0 2.0
has __dict__? False


## Step 2 — Try to create an undeclared attribute


In [68]:
try:
    v.label = "demo"
except AttributeError as exc:
    print(type(exc).__name__ + ":", exc)


AttributeError: 'SlottedVector' object has no attribute 'label' and no __dict__ for setting new attributes


## Design takeaway

`__slots__` can:

- reduce per-instance overhead,
- restrict arbitrary attributes.

But it also changes inheritance and introspection behavior.

It should be used intentionally, not automatically.


# Problem 17 — Dataclass-generated initialization

Many classes have constructors that mostly assign parameters to attributes.

`@dataclass` can generate that boilerplate.


## Step 1 — Compare a manual class


In [69]:
class ManualProduct:
    def __init__(self, name, price, active=True):
        self.name = name
        self.price = price
        self.active = active


## Step 2 — Express the same fields with `@dataclass`


In [70]:
from dataclasses import dataclass


@dataclass
class Product:
    name: str
    price: float
    active: bool = True


In [71]:
product = Product("Keyboard", 99.95)
print(product)


Product(name='Keyboard', price=99.95, active=True)


The dataclass generated an initializer for us.

But we may still need validation.


## Step 3 — Validate after generated initialization with `__post_init__`


In [72]:
from dataclasses import dataclass


@dataclass
class Product:
    name: str
    price: float
    active: bool = True

    def __post_init__(self):
        if not isinstance(self.name, str) or not self.name.strip():
            raise ValueError("name cannot be empty")

        if not isinstance(self.price, (int, float)) or isinstance(self.price, bool):
            raise TypeError("price must be numeric")

        if self.price < 0:
            raise ValueError("price cannot be negative")

        self.name = self.name.strip()
        self.price = float(self.price)


In [73]:
product = Product("  Keyboard  ", 99.95)
print(product)


Product(name='Keyboard', price=99.95, active=True)


## Design takeaway

Dataclasses do not remove initialization.

They automate common initialization mechanics.

`__post_init__` is where we can add validation or normalization after the generated assignments.


# Problem 18 — Mutable defaults in dataclasses

Suppose each `Notebook` needs its own list of tags.

The correct dataclass pattern is:

```python
field(default_factory=list)
```

That calls `list()` separately for every instance.


In [74]:
from dataclasses import dataclass, field


@dataclass
class Notebook:
    title: str
    tags: list = field(default_factory=list)


In [75]:
a = Notebook("Python")
b = Notebook("Databases")

a.tags.append("oop")

print(a)
print(b)
print("same list?", a.tags is b.tags)


Notebook(title='Python', tags=['oop'])
Notebook(title='Databases', tags=[])
same list? False


This solves the same conceptual problem as the classic constructor pattern:

```python
def __init__(self, tags=None):
    self.tags = [] if tags is None else list(tags)
```


# Problem 19 — Dependency injection through initialization

Suppose a report service needs a storage object.

A tightly coupled design would create one specific storage implementation inside the constructor.

Instead, we will inject the dependency.


## Step 1 — Create a test-friendly storage implementation


In [76]:
class MemoryStorage:
    def __init__(self):
        self.files = {}

    def save(self, name, content):
        self.files[name] = content


## Step 2 — Accept the dependency in `__init__`


In [77]:
class ReportService:
    def __init__(self, storage):
        if not hasattr(storage, "save"):
            raise TypeError("storage must provide save")

        if not callable(storage.save):
            raise TypeError("storage.save must be callable")

        self.storage = storage

    def create_report(self, name, content):
        self.storage.save(name, content)


In [78]:
storage = MemoryStorage()
service = ReportService(storage)

service.create_report(
    "summary.txt",
    "Initialization complete.",
)

print(storage.files)


{'summary.txt': 'Initialization complete.'}


## Step 3 — Replace the dependency without changing the service


In [79]:
class ConsoleStorage:
    def save(self, name, content):
        print(f"[SAVE] {name}: {content}")


service = ReportService(ConsoleStorage())
service.create_report("status.txt", "OK")


[SAVE] status.txt: OK


## Design takeaway

Constructor arguments can represent not only data, but also **dependencies**.

This often makes classes easier to:

- test,
- configure,
- extend,
- reuse.


# Problem 20 — Keep expensive work out of `__init__`

Imagine a constructor that immediately:

- reads a huge file,
- opens a network connection,
- contacts a database,
- starts processing.

That makes object construction expensive and surprising.

We will separate initialization from execution.


## Step 1 — Let the constructor establish configuration and state only


In [80]:
class DataImport:
    def __init__(self, source, destination):
        if not source:
            raise ValueError("source is required")

        if not destination:
            raise ValueError("destination is required")

        self.source = source
        self.destination = destination
        self.status = "ready"


## Step 2 — Move the workflow into an explicit method


In [81]:
class DataImport:
    def __init__(self, source, destination):
        if not source:
            raise ValueError("source is required")

        if not destination:
            raise ValueError("destination is required")

        self.source = source
        self.destination = destination
        self.status = "ready"

    def run(self):
        if self.status == "running":
            raise RuntimeError("import is already running")

        self.status = "running"

        result = (
            f"Imported {self.source!r} "
            f"into {self.destination!r}"
        )

        self.status = "finished"
        return result


In [82]:
job = DataImport(
    "customers.csv",
    "analytics_database",
)

print(job.status)
print(job.run())
print(job.status)


ready
Imported 'customers.csv' into 'analytics_database'
finished


## Design takeaway

A lightweight constructor is usually easier to:

- test,
- retry,
- recover from,
- reason about.

Use explicit methods for workflows with significant side effects.


# Problem 21 — Test constructor boundaries

We will create a `Percentage`.

Valid values are:

```text
0 through 100 inclusive
```

We will test both valid and invalid boundaries.


In [83]:
class Percentage:
    def __init__(self, value):
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            raise TypeError("percentage must be numeric")

        if not 0 <= value <= 100:
            raise ValueError("percentage must be between 0 and 100")

        self.value = float(value)


## Step 1 — Create a tiny exception-testing helper

In larger projects, `pytest` is usually preferable.

Here we keep the notebook self-contained.


In [84]:
def assert_raises(expected_exception, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except expected_exception:
        return
    except Exception as exc:
        raise AssertionError(
            f"Expected {expected_exception.__name__}, "
            f"got {type(exc).__name__}"
        ) from exc

    raise AssertionError(
        f"Expected {expected_exception.__name__}, "
        "but no exception was raised"
    )


## Step 2 — Test exact valid boundaries


In [85]:
assert Percentage(0).value == 0.0
assert Percentage(100).value == 100.0

print("valid boundary tests passed")


valid boundary tests passed


## Step 3 — Test just outside the valid range


In [86]:
assert_raises(ValueError, Percentage, -0.01)
assert_raises(ValueError, Percentage, 100.01)
assert_raises(TypeError, Percentage, "50")
assert_raises(TypeError, Percentage, True)

print("failure tests passed")


failure tests passed


## Design takeaway

Boundary tests are especially valuable for constructors.

Always consider:

- minimum valid,
- maximum valid,
- just below valid,
- just above valid,
- wrong type,
- normalized representation.


# Problem 22 — Do not expose owned mutable state directly

We want a `Team`.

The constructor may receive initial members.

The team should own its internal member collection.


## Step 1 — Copy incoming data


In [87]:
class Team:
    def __init__(self, name, members=None):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("name cannot be empty")

        self.name = name.strip()
        self._members = (
            []
            if members is None
            else list(members)
        )


This protects the team from mutation through the caller's original list.

But we could still leak the internal list if we return it directly.


## Step 2 — Return an immutable snapshot


In [88]:
class Team:
    def __init__(self, name, members=None):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("name cannot be empty")

        self.name = name.strip()
        self._members = (
            []
            if members is None
            else list(members)
        )

    @property
    def members(self):
        return tuple(self._members)

    def add_member(self, member):
        if member in self._members:
            raise ValueError("member already exists")

        self._members.append(member)


In [89]:
team = Team("Platform", ["Alice", "Bob"])

print(team.members)

team.add_member("Carla")
print(team.members)


('Alice', 'Bob')
('Alice', 'Bob', 'Carla')


## Step 3 — Confirm callers cannot mutate the exposed tuple


In [90]:
try:
    team.members.append("David")
except AttributeError as exc:
    print(type(exc).__name__ + ":", exc)


AttributeError: 'tuple' object has no attribute 'append'


## Design takeaway

Initialization and encapsulation are connected.

A class that claims ownership of mutable data should think about both:

- how data enters the object,
- how data leaves the object.


# Problem 23 — Refactor an overloaded constructor

Imagine this anti-pattern:

```python
class ImportUsers:
    def __init__(self, filename):
        # read file
        # parse rows
        # validate rows
        # connect to database
        # insert rows
        # send email
```

This constructor is doing an entire workflow.

We will refactor it into explicit stages.


## Step 1 — Constructor only establishes configuration and initial state


In [91]:
class ImportUsers:
    def __init__(self, filename, repository):
        if not filename:
            raise ValueError("filename is required")

        if not hasattr(repository, "save_many"):
            raise TypeError(
                "repository must provide save_many"
            )

        self.filename = filename
        self.repository = repository
        self.rows = []
        self.status = "ready"


## Step 2 — Add an explicit load phase

We will simulate loaded rows rather than reading a real file.


In [92]:
class ImportUsers:
    def __init__(self, filename, repository):
        if not filename:
            raise ValueError("filename is required")

        if not hasattr(repository, "save_many"):
            raise TypeError(
                "repository must provide save_many"
            )

        self.filename = filename
        self.repository = repository
        self.rows = []
        self.status = "ready"

    def load(self, raw_rows):
        if self.status != "ready":
            raise RuntimeError("import is not ready to load")

        self.rows = list(raw_rows)
        self.status = "loaded"


## Step 3 — Add validation as a separate phase


In [93]:
class ImportUsers:
    def __init__(self, filename, repository):
        if not filename:
            raise ValueError("filename is required")

        if not hasattr(repository, "save_many"):
            raise TypeError(
                "repository must provide save_many"
            )

        self.filename = filename
        self.repository = repository
        self.rows = []
        self.status = "ready"

    def load(self, raw_rows):
        if self.status != "ready":
            raise RuntimeError("import is not ready to load")

        self.rows = list(raw_rows)
        self.status = "loaded"

    def validate(self):
        if self.status != "loaded":
            raise RuntimeError("load rows before validation")

        for row in self.rows:
            if not row.get("name"):
                raise ValueError("every row requires a name")

        self.status = "validated"


## Step 4 — Add persistence separately


In [94]:
class MemoryRepository:
    def __init__(self):
        self.saved = []

    def save_many(self, rows):
        self.saved.extend(rows)


class ImportUsers:
    def __init__(self, filename, repository):
        if not filename:
            raise ValueError("filename is required")

        if not hasattr(repository, "save_many"):
            raise TypeError(
                "repository must provide save_many"
            )

        self.filename = filename
        self.repository = repository
        self.rows = []
        self.status = "ready"

    def load(self, raw_rows):
        if self.status != "ready":
            raise RuntimeError("import is not ready to load")

        self.rows = list(raw_rows)
        self.status = "loaded"

    def validate(self):
        if self.status != "loaded":
            raise RuntimeError("load rows before validation")

        for row in self.rows:
            if not row.get("name"):
                raise ValueError("every row requires a name")

        self.status = "validated"

    def save(self):
        if self.status != "validated":
            raise RuntimeError("validate rows before saving")

        self.repository.save_many(self.rows)
        self.status = "saved"


In [95]:
repository = MemoryRepository()
job = ImportUsers("users.csv", repository)

job.load([
    {"name": "Alice"},
    {"name": "Bob"},
])

job.validate()
job.save()

print(job.status)
print(repository.saved)


saved
[{'name': 'Alice'}, {'name': 'Bob'}]


## Design takeaway

The constructor now does one thing well:

> establish a valid `ImportUsers` object.

The rest of the workflow happens explicitly.

That makes failure handling and testing much easier.


# Problem 24 — Capstone: design a robust enrollment model

We will combine many ideas in one small domain model.

We need:

```text
Student
Course
Enrollment
```

We want the constructors to establish strong local invariants, while methods handle later state transitions.


## Part A — Design `Student`

Requirements:

- positive integer `student_id`,
- non-empty name,
- normalized name.


In [96]:
class Student:
    def __init__(self, student_id, name):
        if not isinstance(student_id, int) or isinstance(student_id, bool):
            raise TypeError("student_id must be an integer")

        if student_id <= 0:
            raise ValueError("student_id must be positive")

        if not isinstance(name, str) or not name.strip():
            raise ValueError("name cannot be empty")

        self.student_id = student_id
        self.name = name.strip()

    @classmethod
    def from_dict(cls, data):
        return cls(
            student_id=data["student_id"],
            name=data["name"],
        )

    def __repr__(self):
        return (
            f"Student(student_id={self.student_id}, "
            f"name={self.name!r})"
        )


## Part B — Design `Course`

Requirements:

- course code must be non-empty,
- normalize code to uppercase,
- title must be non-empty,
- capacity must be a positive integer,
- every new course begins with a fresh internal enrollment list.


In [97]:
class Course:
    def __init__(self, code, title, capacity):
        if not isinstance(code, str) or not code.strip():
            raise ValueError("code cannot be empty")

        if not isinstance(title, str) or not title.strip():
            raise ValueError("title cannot be empty")

        if not isinstance(capacity, int) or isinstance(capacity, bool):
            raise TypeError("capacity must be an integer")

        if capacity <= 0:
            raise ValueError("capacity must be positive")

        self.code = code.strip().upper()
        self.title = title.strip()
        self.capacity = capacity
        self._enrollments = []

    @classmethod
    def from_dict(cls, data):
        return cls(
            code=data["code"],
            title=data["title"],
            capacity=data["capacity"],
        )

    @property
    def enrollments(self):
        return tuple(self._enrollments)

    @property
    def enrolled_count(self):
        return len(self._enrollments)


## Part C — Design `Enrollment`

An enrollment is meaningful only when it connects:

- a real `Student`,
- a real `Course`.

We also record the creation time.


In [98]:
from datetime import datetime


class Enrollment:
    def __init__(self, student, course):
        if not isinstance(student, Student):
            raise TypeError("student must be a Student")

        if not isinstance(course, Course):
            raise TypeError("course must be a Course")

        self.student = student
        self.course = course
        self.created_at = datetime.now()

    def __repr__(self):
        return (
            f"Enrollment(student={self.student.student_id}, "
            f"course={self.course.code!r})"
        )


## Part D — Decide where enrollment rules belong

Rules such as:

- no duplicate student,
- do not exceed capacity,

are not merely construction rules for one object.

They describe a **state transition** in the course.

So an explicit method is a good fit:

```python
course.enroll(student)
```


In [99]:
class Course:
    def __init__(self, code, title, capacity):
        if not isinstance(code, str) or not code.strip():
            raise ValueError("code cannot be empty")

        if not isinstance(title, str) or not title.strip():
            raise ValueError("title cannot be empty")

        if not isinstance(capacity, int) or isinstance(capacity, bool):
            raise TypeError("capacity must be an integer")

        if capacity <= 0:
            raise ValueError("capacity must be positive")

        self.code = code.strip().upper()
        self.title = title.strip()
        self.capacity = capacity
        self._enrollments = []

    @classmethod
    def from_dict(cls, data):
        return cls(
            code=data["code"],
            title=data["title"],
            capacity=data["capacity"],
        )

    @property
    def enrollments(self):
        return tuple(self._enrollments)

    @property
    def enrolled_count(self):
        return len(self._enrollments)

    def enroll(self, student):
        if not isinstance(student, Student):
            raise TypeError("student must be a Student")

        duplicate = any(
            enrollment.student.student_id == student.student_id
            for enrollment in self._enrollments
        )

        if duplicate:
            raise ValueError("student is already enrolled")

        if self.enrolled_count >= self.capacity:
            raise OverflowError("course is full")

        enrollment = Enrollment(student, self)
        self._enrollments.append(enrollment)

        return enrollment


## Part E — Build objects from external dictionaries


In [100]:
course = Course.from_dict({
    "code": " py-500 ",
    "title": "Advanced Python Object Model",
    "capacity": 2,
})

students = [
    Student.from_dict({
        "student_id": 1,
        "name": "Ada Lovelace",
    }),
    Student.from_dict({
        "student_id": 2,
        "name": "Grace Hopper",
    }),
    Student.from_dict({
        "student_id": 3,
        "name": "Guido van Rossum",
    }),
]

print(course.code)
print(students)


PY-500
[Student(student_id=1, name='Ada Lovelace'), Student(student_id=2, name='Grace Hopper'), Student(student_id=3, name='Guido van Rossum')]


## Part F — Perform valid enrollments


In [101]:
e1 = course.enroll(students[0])
e2 = course.enroll(students[1])

print(e1)
print(e2)
print("count:", course.enrolled_count)
print("enrollments:", course.enrollments)


Enrollment(student=1, course='PY-500')
Enrollment(student=2, course='PY-500')
count: 2
enrollments: (Enrollment(student=1, course='PY-500'), Enrollment(student=2, course='PY-500'))


## Part G — Test duplicate prevention


In [102]:
try:
    course.enroll(students[0])
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)


ValueError: student is already enrolled


## Part H — Test capacity enforcement


In [103]:
try:
    course.enroll(students[2])
except OverflowError as exc:
    print(type(exc).__name__ + ":", exc)


OverflowError: course is full


## Capstone discussion

This model combines several constructor principles:

### Constructors establish local validity

`Student` validates student identity.

`Course` validates course configuration.

`Enrollment` validates the types it connects.

### Mutable collections are created per instance

Each course starts with its own `_enrollments` list.

### Internal mutable state is protected

The public `enrollments` property returns a tuple.

### Alternate constructors delegate

`from_dict()` converts external data into the normal constructor API.

### State transitions use explicit methods

Enrollment itself belongs in `Course.enroll()` rather than being hidden inside object construction.

That separation keeps initialization predictable.


# Mini-Problem 1 — Normalize before storing

Create a `LanguageCode`.

Rules:

- input must be a string,
- strip whitespace,
- lowercase it,
- require exactly two alphabetic characters.

Try it yourself first.


In [104]:
# Your attempt here


## Solution


In [105]:
class LanguageCode:
    def __init__(self, value):
        if not isinstance(value, str):
            raise TypeError("language code must be a string")

        value = value.strip().lower()

        if len(value) != 2 or not value.isalpha():
            raise ValueError(
                "language code must contain exactly two letters"
            )

        self.value = value


code_en = LanguageCode(" EN ")
print(code_en.value)


en


The order is important:

```text
type check
normalize
validate normalized representation
store
```


# Mini-Problem 2 — Reject rather than silently repair invalid input

Suppose an age cannot be negative.

This may look convenient:

```python
self.age = max(0, age)
```

But if the caller accidentally passes `-20`, silently converting it to `0` hides the bug.

When input violates the model's meaning, an exception is often safer.


In [106]:
class Age:
    def __init__(self, value):
        if not isinstance(value, int) or isinstance(value, bool):
            raise TypeError("age must be an integer")

        if value < 0:
            raise ValueError("age cannot be negative")

        self.value = value


try:
    Age(-20)
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)


ValueError: age cannot be negative


# Mini-Problem 3 — Detect unused cooperative arguments

A cooperative base can catch unsupported keyword arguments instead of silently ignoring them.


In [107]:
class BaseConfig:
    def __init__(self, **kwargs):
        if kwargs:
            raise TypeError(
                f"unused arguments: {sorted(kwargs)}"
            )


class FeatureConfig(BaseConfig):
    def __init__(self, *, enabled, **kwargs):
        self.enabled = bool(enabled)
        super().__init__(**kwargs)


ok = FeatureConfig(enabled=True)
print(ok.__dict__)

try:
    FeatureConfig(
        enabled=True,
        unknown_option=123,
    )
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


{'enabled': True}
TypeError: unused arguments: ['unknown_option']


# Final Review — Constructor Design Checklist

When designing `__init__`, ask:

## 1. What state must exist immediately?

Store the minimum state required for a valid object.

## 2. What input is invalid?

Reject invalid input clearly.

## 3. What should be normalized?

Typical examples:

- whitespace,
- case,
- numeric representation,
- iterable-to-list conversion.

## 4. Am I accidentally sharing mutable state?

Avoid constructor defaults such as:

```python
items=[]
options={}
tags=set()
```

## 5. Does the instance own mutable input?

If yes, consider copying it.

## 6. Is some state derivable?

Prefer a property if storing duplicate state could create inconsistency.

## 7. Are there too many positional options?

Use keyword-only parameters.

## 8. Is this a subclass?

Reuse inherited initialization with `super()`.

## 9. Is multiple inheritance involved?

Use cooperative constructors and respect the MRO.

## 10. Is there another input representation?

Use a classmethod alternate constructor and delegate to `cls(...)`.

## 11. Is expensive I/O happening in `__init__`?

Move it to an explicit workflow method when practical.

## 12. Does object creation itself need customization?

Only then consider `__new__`.

## 13. Is the object an immutable built-in subclass?

Its immutable payload may need to be created in `__new__`.

## 14. Would a dataclass reduce boilerplate?

Use `@dataclass` when it makes the class clearer.

## 15. Have failure paths been tested?

Constructor exceptions and boundary behavior deserve tests too.


# Final Concept Check

Answer these before reading the solutions.

1. Does `__init__` create an instance?
2. Which special method is responsible for creation?
3. What must `__init__` return?
4. Why is `items=[]` dangerous?
5. Why might a constructor copy a caller's list?
6. What does `*` do in a constructor signature?
7. Why can derived values be better as properties?
8. Why should alternate constructors normally call `cls(...)`?
9. What does `super()` really follow?
10. Why can immutable built-in subclasses require `__new__`?
11. What is `__post_init__` used for?
12. Why are expensive side effects often better outside `__init__`?


# Concept Check Solutions

1. **No.** `self` already exists when `__init__` runs.
2. `__new__`.
3. `None`.
4. The same mutable default object can be reused across constructor calls.
5. To give the instance independent ownership and prevent external mutation from changing internal state.
6. Parameters after `*` are keyword-only.
7. Properties remain synchronized because they are computed from source-of-truth state.
8. It reuses the primary constructor and supports subclasses.
9. The class's Method Resolution Order.
10. Their immutable payload must be established during creation.
11. Validation, normalization, or extra setup after dataclass-generated initialization.
12. Explicit workflow methods are easier to test, control, retry, and reason about.


# End

The central principle is:

> A constructor should leave an object in a clear, valid, predictable state.

For ordinary user-defined classes, `__init__` is the main tool.

Use `__new__`, cooperative multiple inheritance, dataclass hooks, `__slots__`, and immutable-subclass techniques only when the design actually calls for them.
